In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from itertools import combinations
from pathlib import Path
from statsmodels.tsa.stattools import adfuller

TRADING_DAYS = 252   # for annualizing Sharpe

In [ ]:
# finding beta
def hedge_ratio(y: pd.Series, x: pd.Series) -> float:
    model = sm.OLS(y, sm.add_constant(x)).fit()
    return float(model.params.iloc[1])   # params = [intercept, slope]

In [ ]:
# ADF test for cointegration and flatness
def engle_granger_pvalue(y: pd.Series, x: pd.Series) -> float:
    df = pd.concat([y, x], axis=1).dropna()
    y_aligned, x_aligned = df.iloc[:, 0], df.iloc[:, 1]   # same dates on both legs

    beta = hedge_ratio(y_aligned, x_aligned)
    spread = y_aligned - beta * x_aligned

    return float(adfuller(spread, regression="c", autolag="AIC")[1])   # [1] is the p-value

In [ ]:
def find_cointegrated_pairs(prices: pd.DataFrame, max_pvalue: float = 0.05) -> pd.DataFrame:
    rows = []

    for a, b in combinations(prices.columns, 2):
        y, x = prices[a], prices[b]
        rows.append({
            "a": a,
            "b": b,
            "pvalue": engle_granger_pvalue(y, x),
            "beta": hedge_ratio(y, x),
        })

    result = pd.DataFrame(rows, columns=["a", "b", "pvalue", "beta"])
    result = result[result["pvalue"] <= max_pvalue]
    return result.sort_values("pvalue").reset_index(drop=True)   # best first

In [ ]:
def _project_root() -> Path:
    # walk up from the cwd until we hit the folder with pytest.ini
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "pytest.ini").exists():
            return candidate
    return here


CACHE_DIR = _project_root() / "data"


def load_prices(tickers: list[str], start: str, end: str, cache: bool = True) -> pd.DataFrame:
    cache_file = CACHE_DIR / f"{'_'.join(sorted(tickers))}_{start}_{end}.csv"

    if cache and cache_file.exists():
        prices = pd.read_csv(cache_file, index_col=0, parse_dates=True)
    else:
        import yfinance as yf

        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        prices = raw["Close"]
        if isinstance(prices, pd.Series):        # a single ticker comes back as a Series
            prices = prices.to_frame(tickers[0])
        if cache:
            CACHE_DIR.mkdir(exist_ok=True)
            prices.to_csv(cache_file)

    return prices.dropna(how="any")   # drop days where any leg is missing

In [ ]:
def compute_spread(y: pd.Series, x: pd.Series, beta: float) -> pd.Series:
    # neither leg is stable on its own, this combination is
    return y - beta * x


def rolling_zscore(spread: pd.Series, window: int = 60) -> pd.Series:
    rolling = spread.rolling(window)                      # window ENDS at t -> only past data
    return (spread - rolling.mean()) / rolling.std()      # first window-1 are NaN


def generate_positions(zscore: pd.Series, entry: float = 2.0, exit: float = 0.5) -> pd.Series:
    positions = np.zeros(len(zscore), dtype=int)
    state = 0     # +1 long the spread, -1 short, 0 flat

    for i, z in enumerate(zscore.to_numpy(dtype=float)):
        if np.isnan(z):
            state = 0                      # warm-up period: stay out
        elif state == 0:
            if z > entry:
                state = -1                 # too wide -> short it
            elif z < -entry:
                state = 1                  # too narrow -> long it
        elif state == 1:
            if z >= -exit:
                state = 0                  # reverted, close out
        else:
            if z <= exit:
                state = 0
        positions[i] = state

    return pd.Series(positions, index=zscore.index, dtype=int)

In [ ]:
def backtest_pair(y: pd.Series, x: pd.Series, beta: float,
                  positions: pd.Series, cost_bps: float = 5.0) -> pd.DataFrame:
    x = x.reindex(y.index)                             # same dates on both legs
    positions = positions.reindex(y.index).fillna(0)

    ret_y = y.pct_change()
    ret_x = x.pct_change()

    lagged = positions.shift(1)                        # decide today, trade tomorrow
    ret = (lagged * (ret_y - ret_x) / 2.0).fillna(0.0)

    turnover = positions.diff()
    turnover.iloc[0] = positions.iloc[0]               # before day 1 we were flat
    cost = turnover.abs() * (cost_bps / 10_000.0)      # bps -> decimal

    ret_net = ret - cost
    equity = (1.0 + ret_net).cumprod()                 # growth of $1

    return pd.DataFrame({"ret": ret, "ret_net": ret_net, "equity": equity}, index=y.index)


def sharpe_ratio(daily_returns: pd.Series) -> float:
    r = daily_returns.dropna()
    if len(r) < 2:
        return 0.0
    sd = r.std(ddof=1)
    if sd == 0 or np.isnan(sd):
        return 0.0                                      # avoid div by zero
    return float(r.mean() / sd * np.sqrt(TRADING_DAYS))


def max_drawdown(equity: pd.Series) -> float:
    eq = equity.dropna()
    if eq.empty:
        return 0.0
    drawdown = eq / eq.cummax() - 1.0                   # distance below the running peak
    return float(drawdown.min())


def summarize(result: pd.DataFrame, positions: pd.Series) -> dict:
    equity = result["equity"].dropna()
    total_return = float(equity.iloc[-1] - 1.0) if not equity.empty else 0.0

    pos = positions.reindex(result.index).fillna(0)
    prev = pos.shift(1).fillna(0)
    n_round_trips = int(((prev != 0) & (pos == 0)).sum())   # nonzero -> flat transitions

    return {
        "total_return": total_return,
        "sharpe": sharpe_ratio(result["ret_net"]),
        "max_drawdown": max_drawdown(result["equity"]),
        "n_round_trips": n_round_trips,
    }

In [ ]:
# skip-on-import
UNIVERSE = ["XOM", "CVX", "COP", "JPM", "BAC", "WFC", "GS", "MS", "ICE", "CME", "V", "MA"]

FORMATION = ("2018-01-01", "2021-12-31")   # search for pairs here
TRADING   = ("2022-01-01", "2024-12-31")   # trade them here, out of sample

formation_prices = load_prices(UNIVERSE, *FORMATION)
print(f"{formation_prices.shape[0]} days x {formation_prices.shape[1]} tickers")

n = len(UNIVERSE)
print(f"{n} tickers -> {n*(n-1)//2} pair tests -> ~{n*(n-1)//2*0.05:.1f} expected false positives at 5%")

pairs = find_cointegrated_pairs(formation_prices, max_pvalue=0.05)
pairs

In [ ]:
# skip-on-import
best = pairs.iloc[0]
a, b = best["a"], best["b"]

trading_prices = load_prices([a, b], *TRADING)
y, x = trading_prices[a], trading_prices[b]

beta = hedge_ratio(formation_prices[a], formation_prices[b])   # formation window ONLY

spread = compute_spread(y, x, beta)
z = rolling_zscore(spread, window=60)
pos = generate_positions(z, entry=2.0, exit=0.5)
result = backtest_pair(y, x, beta, pos, cost_bps=5.0)

print(f"{a}/{b}  beta={beta:.2f}")
for k, v in summarize(result, pos).items():
    print(f"  {k:>15}: {v:.4f}" if isinstance(v, float) else f"  {k:>15}: {v}")

In [ ]:
# skip-on-import
# re-test each survivor on the trading window -- accidents show up as a bad OOS p
rows = []
for _, row in pairs.iterrows():
    pa, pb = row["a"], row["b"]
    t = load_prices([pa, pb], *TRADING)
    bt = hedge_ratio(formation_prices[pa], formation_prices[pb])
    zz = rolling_zscore(compute_spread(t[pa], t[pb], bt), window=60)
    pp = generate_positions(zz, entry=2.0, exit=0.5)
    stats = summarize(backtest_pair(t[pa], t[pb], bt, pp, cost_bps=5.0), pp)
    rows.append({
        "pair": f"{pa}/{pb}",
        "p_in_sample": row["pvalue"],
        "p_out_sample": engle_granger_pvalue(t[pa], t[pb]),
        **stats,
    })

pd.DataFrame(rows).sort_values("sharpe", ascending=False).reset_index(drop=True)

In [ ]:
# skip-on-import
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

z.plot(ax=axes[0], title=f"{a}/{b} rolling z-score")
axes[0].axhline(2, ls="--", c="gray")
axes[0].axhline(-2, ls="--", c="gray")
axes[0].axhline(0, ls="-", c="lightgray", lw=0.8)

result["equity"].plot(ax=axes[1], title="Equity curve (net of costs)")
axes[1].axhline(1.0, ls="--", c="gray", lw=0.8)

plt.tight_layout()
plt.savefig("backtest.png", dpi=120)
plt.show()